# 03 - Threshold analysis
Illustrative cost-sensitive tuning: FN cost=10, FP cost=1. Default 0.5 is not assumed optimal.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import joblib, pandas as pd
from sklearn.model_selection import train_test_split
from src.config import RAW_FEATURES, TARGET, RANDOM_SEED
from src.train import threshold_report

df = pd.read_csv('../data/ai4i2020.csv').dropna(subset=RAW_FEATURES + [TARGET])
X = df[RAW_FEATURES]; y = df[TARGET].astype(int)
_, X_te, _, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)
pipe = joblib.load('../models/predictive_maintenance_pipeline.pkl')
proba = pipe.predict_proba(X_te[RAW_FEATURES])[:, 1]
for row in threshold_report(y_te, proba):
    cost = 10 * row['false_negatives'] + row['false_positives']
    print(f"t={row['threshold']:.2f} recall={row['recall']:.3f} precision={row['precision']:.3f} "
          f"f1={row['f1']:.3f} FN={row['false_negatives']} FP={row['false_positives']} cost={cost}")